In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os, time
import pickle
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp

In [ ]:
# !pip install ecos
# !pip install scs

In [6]:
import tempfile

def atomic_pickle_dump(obj, final_path, verbose=True):
    dir_name = os.path.dirname(final_path)

    if verbose:
        print(f"[atomic_pickle_dump] Saving to temporary file in: {dir_name}")

    with tempfile.NamedTemporaryFile(
        dir=dir_name, delete=False, suffix=".tmp"
    ) as tmp:
        pickle.dump(obj, tmp)
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp_path = tmp.name

        if verbose:
            print(f"[atomic_pickle_dump] ✓ Written and flushed temp file: {tmp_path}")

    os.replace(tmp_path, final_path)

    if verbose:
        print(f"[atomic_pickle_dump] ✅ Successfully promoted temp file → {final_path}")

# Algorithm

In [7]:
# --- Hàm xây dựng Ma trận A (từ Ảnh 2) ---
def build_A_matrix(n_goods, n_slots):
    """Tạo ma trận ánh xạ Goods -> Slots"""
    A = np.zeros((n_slots, n_goods))
    for j in range(n_goods):
        A[j % n_slots, j] = 1.0
    return A

In [8]:
# --- Hàm buyer best response for each user used in SPDS
def buyer_best_response_cvx(v_i, p, A, q_i, B_i, utility_type="Linear"):
    """
    Phiên bản siêu tốc của buyer_best_response.
    Sử dụng Closed-form solution cho mọi hàm utility.
    KHÔNG DÙNG CVXPY.
    """
    # 1. Tính giá hiệu dụng (Effective Price)
    # p: (M,), A.T @ q_i: (M,)
    effective_price = p + (A.T @ q_i)

    # An toàn: Đảm bảo giá > 0 để tránh chia cho 0
    safe_price = np.maximum(effective_price, 1e-9)

    m_goods = len(v_i)
    x = np.zeros(m_goods)

    # =========================================================
    # 1. LINEAR UTILITY: U = sum(v * x)
    # =========================================================
    if utility_type == "Linear":
        # Chiến thuật: Bang-per-buck (Mua tất tay món hời nhất)
        bang_per_buck = v_i / safe_price
        best_idx = np.argmax(bang_per_buck)
        x[best_idx] = B_i / safe_price[best_idx]

    # =========================================================
    # 2. COBB-DOUGLAS: U = prod(x^alpha) hoặc sum(alpha * log(x))
    # =========================================================
    elif utility_type == "Cobb-Douglas":
        # Chuẩn hóa alpha (Bắt buộc để thỏa mãn Budget constraint)
        sum_v = np.sum(v_i)
        if sum_v > 0:
            alpha = v_i / sum_v
        else:
            alpha = np.zeros_like(v_i) # Tránh lỗi nếu v toàn 0

        # Công thức: Chi tiêu đúng tỷ lệ alpha
        # x_i = (alpha_i * Budget) / Price_i
        x = (alpha * B_i) / safe_price

    # =========================================================
    # 3. LEONTIEF: U = min(x / v)
    # =========================================================
    elif utility_type == "Leontief":
        # Chiến thuật: Mua theo tỷ lệ cố định của v
        # Giá của 1 combo chuẩn = sum(v_i * p_i)
        cost_of_one_bundle = np.dot(v_i, safe_price)

        if cost_of_one_bundle > 0:
            # Số lượng combo mua được
            num_bundles = B_i / cost_of_one_bundle
            x = num_bundles * v_i
        else:
            x = np.zeros(m_goods)

    # =========================================================
    # 4. CES UTILITY: U = (sum v * x^rho)^(1/rho)
    # =========================================================
    elif utility_type == "CES_8":
        # CẤU HÌNH rho (m) TẠI ĐÂY
        # Trong code cũ bạn để m = 1/2
        rho = 0.5

        # --- CASE A: CONCAVE (rho < 1, rho != 0) ---
        # Đây là trường hợp thay thế (Substitute) -> Mua nhiều loại
        if rho < 1:
            # Tính Sigma (Elasticity of Substitution)
            # sigma = 1 / (1 - rho)
            sigma = 1.0 / (1.0 - rho)

            # Tính phần tử tỷ lệ: Term_i = (v_i / p_i)^sigma
            # Dùng np.maximum cho v_i để tránh v=0 gây lỗi log hoặc mũ âm
            term = np.power(v_i / safe_price, sigma)

            # Tính mẫu số chung: Sum (p_j * term_j)
            denom = np.dot(safe_price, term)

            if denom > 0:
                # x_i = (B * term_i) / denom
                x = (B_i * term) / denom
            else:
                 # Fallback nếu v=0 hết
                 x = np.zeros(m_goods)

        # --- CASE B: CONVEX (rho > 1) ---
        # Đây là trường hợp "Winner Takes All" giống Linear
        else:
            # So sánh tỷ lệ: v^(1/rho) / p
            v_transformed = np.power(v_i, 1.0/rho)
            bang_per_buck = v_transformed / safe_price

            best_idx = np.argmax(bang_per_buck)
            x[best_idx] = B_i / safe_price[best_idx]

    return x

# --- GROUND TRUTH OPTIMAL RESULT
def solve_centralized_optimal_fair(valuations, budgets, supply_s, b_mat, A, utility_type="Linear"):
    """
    Returns:
        optimal_value (float): Giá trị hàm mục tiêu tối ưu.
        optimal_X (np.ndarray): Ma trận phân bổ tối ưu (N x M).
    """
    n, m = valuations.shape
    X = cp.Variable((n, m), nonneg=True)

    constraints = [
        cp.sum(X, axis=0) <= supply_s
    ]
    for i in range(n):
        constraints.append(A @ X[i] <= b_mat[i])

    # --- XÂY DỰNG OBJECTIVE ---
    if utility_type == "Linear":
        utilities = cp.sum(cp.multiply(valuations, X), axis=1)
        primal_utility = cp.sum(cp.multiply(budgets, cp.log(utilities + 1e-12)))

    elif utility_type == "Cobb-Douglas":
        eps = 1e-12
        alpha = valuations / (np.sum(valuations, axis=1, keepdims=True))
        log_utilities = cp.sum(cp.multiply(alpha, cp.log(X + eps)), axis=1)
        primal_utility = cp.sum(cp.multiply(budgets, log_utilities))

    elif utility_type == "Leontief":
        # Lưu ý: Leontief trong CVXPY có thể phức tạp/chậm với quy mô lớn
        inv_valuations = np.divide(
            1.0,
            valuations,
            out=np.full_like(valuations, 1e30),
            where=(valuations > 0)
        )

        # 2. Nhân X với ma trận nghịch đảo hằng số này
        # ratio_matrix[i, j] = X[i, j] * (1 / v[i, j])
        weighted_X = cp.multiply(X, inv_valuations)

        # 3. Lấy min theo hàng
        utilities = cp.min(weighted_X, axis=1)

        # 4. Tính hàm mục tiêu
        primal_utility = cp.sum(cp.multiply(budgets, cp.log(utilities + 1e-12)))

    elif utility_type == "CES_8":
        # Nếu chạy Solver tập trung, m NÊN nhỏ hơn hoặc bằng 1 (ví dụ 0.5 hoặc -1).
        # Nếu đặt m = 8, Solver ECOS/SCS sẽ KHÔNG giải được (báo lỗi DCPError).
        eps = 1e-9
        m_ces = 1/2

        # Công thức: log_util = (1/m) * log( sum( v * x^m ) )
        # Tính tổng trọng số lũy thừa theo hàng (axis=1)
        inner_term = cp.sum(cp.multiply(valuations, cp.power(X + eps, m_ces)), axis=1)

        # Logarit hóa hàm mục tiêu
        log_utilities = (1.0 / m_ces) * cp.log(inner_term)

        primal_utility = cp.sum(cp.multiply(budgets, log_utilities))

    objective = cp.Maximize(primal_utility)
    prob = cp.Problem(objective, constraints)

    print(f"--- Solving Centralized Fair Problem ({utility_type}) ---")
    try:
        prob.solve(solver=cp.ECOS, verbose=False)
    except:
        prob.solve(solver=cp.SCS, verbose=False)

    # TRẢ VỀ: (Giá trị mục tiêu, Ma trận X tối ưu)
    return prob.value, X.value

In [9]:
def full_grad_descent(
    supply_s, capacity_b, A, valuations, budgets,
    p0, q0, lr_p=0.02, lr_q=0.02, num_iters=1000000,
    log_freq=250, seed=42, eps=0.00001):
    """
    Full primal-dual gradient descent with capacity constraints
    """

    np.random.seed(seed)

    n, m = valuations.shape

    p = p0.astype(float).copy()          # goods prices (m,)
    q = q0.astype(float).copy()          # capacity prices (n,T)

    # --- BƯỚC 1: KHỞI TẠO TUYỆT ĐỐI (X0 = 100) ---
    X = np.ones((n, m)) * 1
    util =  np.zeros(n)
    for j in range(n):
        util[j] = valuations[j] @ X[j]

    # --- 2. GHI LOG TRẠNG THÁI BAN ĐẦU (t=0) ---
    # Khởi tạo lịch sử
    num_logs = num_iters // log_freq + 1
    obj_hist = np.empty(num_logs)
    time_hist = np.empty(num_logs)
    log_idx = 0

    # Tính Objective tại điểm xuất phát (lúc này chưa ai mua gì)
    term_p = np.sum(p * supply_s)     # p * Supply
    term_q = np.sum(q * capacity_b)            # q * Capacity
    term_u = np.sum(budgets * np.log(util + 1e-12)) # Utility
    obj_0 = term_p + term_q + term_u - np.sum(budgets)

    obj_hist[log_idx] = obj_0
    time_hist[log_idx] = 0.0
    log_idx += 1

    print(f"--- Start GD ({num_iters} iterations) | Initial Obj: {obj_0:.2f} ---")

    # Easier for final allocation
    X = np.array([buyer_best_response_cvx(valuations[j], p, A, q[j], budgets[j])
                for j in range(n)])


    try:
        for t in range(1, num_iters + 1):
            elapsed = time.perf_counter() - t0

            # ---------------------------
            # Buyer's best response
            # ---------------------------
            X = np.array([buyer_best_response_cvx(valuations[j], p, A, q[j], budgets[j])
                for j in range(n)])
            util = np.sum(valuations * X, axis=1)

            # ---------------------------
            # Dual gradients
            # ---------------------------
            g_p = X.sum(axis=0) - supply_s   # (1, m)
            g_q = (A @ X.T).T - capacity_b  # (n,T)

            # Step sizes
            alpha = lr_p / np.sqrt(t)
            beta  = lr_q / np.sqrt(t)

            # Dual updates
            p = np.maximum(p + alpha * g_p, 0.0)
            q = np.maximum(q + beta  * g_q, 0.0)

            term_u = np.sum(budgets * np.log(util + 1e-12))
            term_p = np.sum(p * supply_s)
            term_q = np.sum(q * capacity_b)
            obj = term_u + term_p + term_q - np.sum(budgets)

            # ---------------------------
            # Logging
            # ---------------------------
            if t % log_freq == 0:
                # Update history
                obj_hist[log_idx] = obj
                time_hist[log_idx] = elapsed

                # Update log_idx in numpy array
                log_idx += 1

                # ---------- Termination conditions ----------
                ratio = abs((obj_hist[log_idx - 1] - obj_hist[log_idx - 2]) / obj_hist[log_idx - 2])
                if ratio <= eps:
                    print(f"DPDS Iterations: {t}")
                    print(f"Objective value: {obj:.4f}")
                    print(f"DPDS Algorithm finsished at epsilon difference {eps} in {elapsed:.2f} s")
                    break

    except KeyboardInterrupt:
        print("🛑 Interrupted by user")

    except Exception as e:
        print(f"💥 Crash: {e}")

    return obj_hist[:log_idx], time_hist[:log_idx], X.copy(), p, q

# 10u_24i

In [ ]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_10u_24i"

# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except Exception as e:
    print(f"⚠️  Lỗi khi tải file: {e}")
    print("Sử dụng ma trận valuations ngẫu nhiên...")
    valuations = np.random.rand(10, 24) + 0.5
valuations /= 10

n_buyers, n_goods = valuations.shape
n_slots = 4
budgets = np.full(n_buyers, 1.0)
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
num_iters = 10000000
log_freq  = 10
time_limit = 7200
current_lr = 0.02

np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

✓ Valuations Shape: (10, 24)

Thiết lập hoàn tất. (n_buyers=10, n_goods=24)


In [ ]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
      valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN DPDS ---
t0 = time.perf_counter()

obj_full, time_full, full_allocation, p_new, q_new = full_grad_descent(
        supply_s, capacity_b, A_matrix, valuations, budgets, p0, q0,
        lr_p=current_lr, lr_q=current_lr, num_iters=num_iters, log_freq=log_freq)
t_full = time.perf_counter() - t0

print(f"\nGD finished in {t_full:.2f} s")
print(f"Full gradient finished in {t_full:.2f} s")
print(f"Speed-up = {t_full / t_full:.1f}× (wall-clock)\n")

# 1. Gom TẤT CẢ các biến bạn muốn lưu vào một dictionary
all_results = {
    'obj_hist': obj_full,
    'time_log': time_full,
    'alloc_final': full_allocation,
    'total_time': t_full,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'seed': 42,
        'obj_tol': 1e-2,
        'num_iters' : len(obj_full) * log_freq,
        'log_freq' : log_freq
    },
    'data': { # Lưu cả tham số để đối chiếu
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'budgets' : budgets,
        'supply_s'  : supply_s,
        'capacity_b': capacity_b,
        'p_new'      : p_new,
        'q_new'      : q_new,
        'A_matrix': A_matrix
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_10u_24i.pkl'
with open(save_path_pickle, 'wb') as file_handle:
     pickle.dump(all_results, file_handle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (4, 24)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -63.9093
★ Optimal Allocation Shape (X*): (10, 24)
--- Start GD (10000000 iterations) | Initial Obj: 166.35 ---
DPDS Iterations: 4330
Objective value: -63.6066
DPDS Algorithm finsished at epsilon difference 0.001 in 0.70 s

GD finished in 0.70 s
Full gradient finished in 0.70 s
Speed-up = 1.0× (wall-clock)

Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_10u_24i.pkl


# 50u_60i

In [ ]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_50u_60i"

# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except Exception as e:
    print(f"⚠️  Lỗi khi tải file: {e}")
    print("Sử dụng ma trận valuations ngẫu nhiên...")
    valuations = np.random.rand(50, 60) + 0.5
valuations /= 10

n_buyers, n_goods = valuations.shape
n_slots = 10
budgets = np.full(n_buyers, 1.0)
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
num_iters = 10000000
log_freq  = 10
time_limit = 7200
current_lr = 0.02

np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

✓ Valuations Shape: (50, 60)

Thiết lập hoàn tất. (n_buyers=50, n_goods=60)


In [ ]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
      valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN DPDS ---
t0 = time.perf_counter()

obj_full, time_full, full_allocation, p_new, q_new = full_grad_descent(
        supply_s, capacity_b, A_matrix, valuations, budgets, p0, q0,
        lr_p=current_lr, lr_q=current_lr, num_iters=num_iters, log_freq=log_freq)
t_full = time.perf_counter() - t0

print(f"\nGD finished in {t_full:.2f} s")
print(f"Full gradient finished in {t_full:.2f} s")
print(f"Speed-up = {t_full / t_full:.1f}× (wall-clock)\n")

# 1. Gom TẤT CẢ các biến bạn muốn lưu vào một dictionary
all_results = {
    'obj_hist': obj_full,
    'time_log': time_full,
    'alloc_final': full_allocation,
    'total_time': t_full,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'seed': 42,
        'obj_tol': 1e-2,
        'num_iters' : len(obj_full) * log_freq,
        'log_freq' : log_freq
    },
    'data': { # Lưu cả tham số để đối chiếu
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'budgets' : budgets,
        'supply_s'  : supply_s,
        'capacity_b': capacity_b,
        'p0'      : p0,
        'q0'      : q0,
        'A_matrix': A_matrix
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_50u_60i.pkl'
with open(save_path_pickle, 'wb') as file_handle:
     pickle.dump(all_results, file_handle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (10, 60)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -960.1566
★ Optimal Allocation Shape (X*): (50, 60)
--- Start GD (10000000 iterations) | Initial Obj: 1201.57 ---
DPDS Iterations: 940
Objective value: -949.8793
DPDS Algorithm finsished at epsilon difference 0.001 in 1.15 s

GD finished in 1.16 s
Full gradient finished in 1.16 s
Speed-up = 1.0× (wall-clock)

Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_50u_60i.pkl


# 100u_150i

In [10]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_100u_150i"


# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(100, 150) + 0.1
valuations /= 10

# --- CONFIGURATION
n_buyers, n_goods = valuations.shape
n_slots = 25
budgets = np.full(n_buyers, 1.0)
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
num_iters = 10000000
log_freq  = 10
current_lr = 0.0002

np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

✓ Valuations Shape: (100, 150)

Thiết lập hoàn tất. (n_buyers=100, n_goods=150)


In [11]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
      valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN DPDS ---
t0 = time.perf_counter()

obj_full, time_full, full_allocation, p_new, q_new = full_grad_descent(
        supply_s, capacity_b, A_matrix, valuations, budgets, p0, q0,
        num_iters=num_iters, lr_p=current_lr, lr_q=current_lr, log_freq=log_freq)
t_full = time.perf_counter() - t0

print(f"\nGD finished in {t_full:.2f} s")
print(f"Full gradient finished in {t_full:.2f} s")
print(f"Speed-up = {t_full / t_full:.1f}× (wall-clock)\n")

# 1. Gom TẤT CẢ các biến bạn muốn lưu vào một dictionary
all_results = {
    'obj_hist': obj_full,
    'time_log': time_full,
    'alloc_final': full_allocation,
    'total_time': t_full,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'obj_tol': 1e-2,
        'num_iters' : len(obj_full) * log_freq,
        'num_iters' : num_iters,
        'log_freq' : log_freq
    },
    'data': { # Lưu cả tham số để đối chiếu
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'budgets' : budgets,
        'supply_s'  : supply_s,
        'capacity_b': capacity_b,
        'p0'      : p0,
        'q0'      : q0,
        'A_matrix': A_matrix
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_100u_150i.pkl'
with open(save_path_pickle, 'wb') as file_handle:
     pickle.dump(all_results, file_handle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (25, 150)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -2593.4031
★ Optimal Allocation Shape (X*): (100, 150)
--- Start GD (10000000 iterations) | Initial Obj: 110.76 ---
DPDS Iterations: 80
Objective value: -2586.7572
DPDS Algorithm finsished at epsilon difference 1e-05 in 0.12 s

GD finished in 0.13 s
Full gradient finished in 0.13 s
Speed-up = 1.0× (wall-clock)

Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_100u_150i.pkl


# 200u_240i

In [12]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_200u_240i"


# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(200, 240) + 0.1
valuations /= 10

# --- CONFIGURATION
n_buyers, n_goods = valuations.shape
n_slots = 40
budgets = np.full(n_buyers, 1.0)
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
num_iters = 10000000
log_freq  = 10
current_lr = 0.0002

np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

✓ Valuations Shape: (200, 240)

Thiết lập hoàn tất. (n_buyers=200, n_goods=240)


In [13]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
      valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN DPDS ---
t0 = time.perf_counter()

obj_full, time_full, full_allocation, p_new, q_new = full_grad_descent(
        supply_s, capacity_b, A_matrix, valuations, budgets,
        p0, q0, num_iters=num_iters, lr_p=current_lr, lr_q=current_lr,
        log_freq=log_freq)
t_full = time.perf_counter() - t0

print(f"\nGD finished in {t_full:.2f} s")
print(f"Full gradient finished in {t_full:.2f} s")
print(f"Speed-up = {t_full / t_full:.1f}× (wall-clock)\n")

# 1. Gom TẤT CẢ các biến bạn muốn lưu vào một dictionary
all_results = {
    'obj_hist': obj_full,
    'time_log': time_full,
    'alloc_final': full_allocation,
    'total_time': t_full,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'seed': 42,
        'obj_tol': 1e-2,
        'num_iters' : len(obj_full) * log_freq,
        'log_freq' : log_freq
    },
    'data': { # Lưu cả tham số để đối chiếu
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'budgets' : budgets,
        'supply_s'  : supply_s,
        'capacity_b': capacity_b,
        'p0'      : p0,
        'q0'      : q0,
        'A_matrix': A_matrix
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_200u_240i.pkl'
with open(save_path_pickle, 'wb') as file_handle:
     pickle.dump(all_results, file_handle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (40, 240)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -6555.3234
★ Optimal Allocation Shape (X*): (200, 240)
--- Start GD (10000000 iterations) | Initial Obj: -204.52 ---
DPDS Iterations: 300
Objective value: -6542.6163
DPDS Algorithm finsished at epsilon difference 1e-05 in 2.58 s

GD finished in 2.60 s
Full gradient finished in 2.60 s
Speed-up = 1.0× (wall-clock)

Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_200u_240i.pkl


# 300u_360i

In [14]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_300u_360i"


# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(300, 360) + 0.1
valuations /= 10

# --- CONFIGURATION
n_buyers, n_goods = valuations.shape
n_slots = 60
budgets = np.full(n_buyers, 1.0)
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
num_iters = 10000000
log_freq  = 10
current_lr = 0.0002

np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

✓ Valuations Shape: (300, 360)

Thiết lập hoàn tất. (n_buyers=300, n_goods=360)


In [15]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
      valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN DPDS ---
t0 = time.perf_counter()

obj_full, time_full, full_allocation, p_new, q_new = full_grad_descent(
        supply_s, capacity_b, A_matrix, valuations, budgets,
        p0, q0, num_iters=num_iters, lr_p=current_lr, lr_q=current_lr,
        log_freq=log_freq)
t_full = time.perf_counter() - t0

print(f"\nGD finished in {t_full:.2f} s")
print(f"Full gradient finished in {t_full:.2f} s")
print(f"Speed-up = {t_full / t_full:.1f}× (wall-clock)\n")

# 1. Gom TẤT CẢ các biến bạn muốn lưu vào một dictionary
all_results = {
    'obj_hist': obj_full,
    'time_log': time_full,
    'alloc_final': full_allocation,
    'total_time': t_full,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'seed': 42,
        'obj_tol': 1e-2,
        'num_iters' : len(obj_full) * log_freq,
        'log_freq' : log_freq
    },
    'data': { # Lưu cả tham số để đối chiếu
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'budgets' : budgets,
        'supply_s'  : supply_s,
        'capacity_b': capacity_b,
        'p0'      : p0,
        'q0'      : q0,
        'A_matrix': A_matrix
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_300u_360i.pkl'
with open(save_path_pickle, 'wb') as file_handle:
     pickle.dump(all_results, file_handle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (60, 360)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -11024.8890
★ Optimal Allocation Shape (X*): (300, 360)
--- Start GD (10000000 iterations) | Initial Obj: -278.60 ---
DPDS Iterations: 130
Objective value: -10928.6401
DPDS Algorithm finsished at epsilon difference 1e-05 in 1.26 s

GD finished in 1.28 s
Full gradient finished in 1.28 s
Speed-up = 1.0× (wall-clock)

Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_300u_360i.pkl


# 400u_480i

In [16]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_400u_480i"


# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(400, 480) + 0.1
valuations /= 10

# --- CONFIGURATION
n_buyers, n_goods = valuations.shape
n_slots = 80
budgets = np.full(n_buyers, 1.0)
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
num_iters = 10000000
log_freq  = 10
current_lr = 0.0002

np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

✓ Valuations Shape: (400, 480)

Thiết lập hoàn tất. (n_buyers=400, n_goods=480)


In [17]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
      valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN DPDS ---
t0 = time.perf_counter()

obj_full, time_full, full_allocation, p_new, q_new = full_grad_descent(
        supply_s, capacity_b, A_matrix, valuations, budgets,
        p0, q0, num_iters=num_iters, lr_p=current_lr, lr_q=current_lr,
        log_freq=log_freq)
t_full = time.perf_counter() - t0

print(f"\nGD finished in {t_full:.2f} s")
print(f"Full gradient finished in {t_full:.2f} s")
print(f"Speed-up = {t_full / t_full:.1f}× (wall-clock)\n")

# 1. Gom TẤT CẢ các biến bạn muốn lưu vào một dictionary
all_results = {
    'obj_hist': obj_full,
    'time_log': time_full,
    'alloc_final': full_allocation,
    'total_time': t_full,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'seed': 42,
        'obj_tol': 1e-2,
        'num_iters' : len(obj_full) * log_freq,
        'log_freq' : log_freq
    },
    'data': { # Lưu cả tham số để đối chiếu
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'budgets' : budgets,
        'supply_s'  : supply_s,
        'capacity_b': capacity_b,
        'p0'      : p0,
        'q0'      : q0,
        'A_matrix': A_matrix
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_400u_480i.pkl'
with open(save_path_pickle, 'wb') as file_handle:
     pickle.dump(all_results, file_handle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (80, 480)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -15871.5403
★ Optimal Allocation Shape (X*): (400, 480)
--- Start GD (10000000 iterations) | Initial Obj: -384.52 ---
DPDS Iterations: 210
Objective value: -15683.7140
DPDS Algorithm finsished at epsilon difference 1e-05 in 3.33 s

GD finished in 3.34 s
Full gradient finished in 3.34 s
Speed-up = 1.0× (wall-clock)

Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_400u_480i.pkl


# 500u_600i

In [18]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_500u_600i"


# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(500, 600) + 0.1
valuations /= 10

# --- CONFIGURATION
n_buyers, n_goods = valuations.shape
n_slots = 100
budgets = np.full(n_buyers, 1.0)
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
num_iters = 10000000
log_freq  = 10
current_lr = 0.0002

np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

✓ Valuations Shape: (500, 600)

Thiết lập hoàn tất. (n_buyers=500, n_goods=600)


In [19]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
      valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN DPDS ---
t0 = time.perf_counter()

obj_full, time_full, full_allocation, p_new, q_new = full_grad_descent(
        supply_s, capacity_b, A_matrix, valuations, budgets,
        p0, q0, num_iters=num_iters, lr_p=current_lr, lr_q=current_lr,
        log_freq=log_freq, eps=0.0001)
t_full = time.perf_counter() - t0

print(f"\nGD finished in {t_full:.2f} s")
print(f"Full gradient finished in {t_full:.2f} s")
print(f"Speed-up = {t_full / t_full:.1f}× (wall-clock)\n")

# 1. Gom TẤT CẢ các biến bạn muốn lưu vào một dictionary
all_results = {
    'obj_hist': obj_full,
    'time_log': time_full,
    'alloc_final': full_allocation,
    'total_time': t_full,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'seed': 42,
        'obj_tol': 1e-2,
        'num_iters' : len(obj_full) * log_freq,
        'log_freq' : log_freq
    },
    'data': { # Lưu cả tham số để đối chiếu
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'budgets' : budgets,
        'supply_s'  : supply_s,
        'capacity_b': capacity_b,
        'p_new'      : p_new,
        'q_new'      : q_new,
        'A_matrix': A_matrix
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_500u_600i.pkl'
with open(save_path_pickle, 'wb') as file_handle:
     pickle.dump(all_results, file_handle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (100, 600)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -20971.4591
★ Optimal Allocation Shape (X*): (500, 600)
--- Start GD (10000000 iterations) | Initial Obj: -491.74 ---
DPDS Iterations: 250
Objective value: -20694.4992
DPDS Algorithm finsished at epsilon difference 0.0001 in 6.06 s

GD finished in 6.09 s
Full gradient finished in 6.09 s
Speed-up = 1.0× (wall-clock)

Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_500u_600i.pkl


# 600u_720i

In [20]:
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_600u_720i"


# --- Code của bạn bắt đầu từ đây ---
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(600, 720) + 0.1
valuations /= 10

# --- CONFIGURATION
n_buyers, n_goods = valuations.shape
n_slots = 120
budgets = np.full(n_buyers, 1.0)
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
q0 = np.zeros((n_buyers, n_slots))
num_iters = 10000000
log_freq  = 10
current_lr = 0.0002

np.random.seed(1)
budgets = np.array([10] * n_buyers)
supply_s = np.random.uniform(5, 10, n_goods)
capacity_b = np.random.uniform(1, 4, (n_buyers, n_slots))

print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)

Thiết lập hoàn tất. (n_buyers=600, n_goods=720)


In [21]:
# --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
if n_goods % n_slots == 0:
    A_matrix = build_A_matrix(n_goods, n_slots)
else:
    A_matrix = np.zeros((n_slots, n_goods))
print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

# --- OPTIMAL DUAL ALLOCATION
target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
      valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
if target_optimal_X is not None:
    print(f"★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

# --- RUN DPDS ---
t0 = time.perf_counter()

obj_full, time_full, full_allocation, p_new, q_new = full_grad_descent(
        supply_s, capacity_b, A_matrix, valuations, budgets,
        p0, q0, num_iters=num_iters, lr_p=current_lr, lr_q=current_lr,
        log_freq=log_freq, eps=0.0001)
t_full = time.perf_counter() - t0

print(f"\nGD finished in {t_full:.2f} s")
print(f"Full gradient finished in {t_full:.2f} s")
print(f"Speed-up = {t_full / t_full:.1f}× (wall-clock)\n")

# 1. Gom TẤT CẢ các biến bạn muốn lưu vào một dictionary
all_results = {
    'obj_hist': obj_full,
    'time_log': time_full,
    'alloc_final': full_allocation,
    'total_time': t_full,
    'config': {
        'lr_p': current_lr,
        'lr_q': current_lr,
        'seed': 42,
        'obj_tol': 1e-2,
        'num_iters' : len(obj_full) * log_freq,
        'log_freq' : log_freq
    },
    'data': { # Lưu cả tham số để đối chiếu
        'n_buyers' : n_buyers,
        'n_goods' : n_goods,
        'budgets' : budgets,
        'supply_s'  : supply_s,
        'capacity_b': capacity_b,
        'p_new'      : p_new,
        'q_new'      : q_new,
        'A_matrix': A_matrix
    }
}

# 2. Định nghĩa đường dẫn và lưu file
save_path_pickle = '/content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_600u_720i.pkl'
with open(save_path_pickle, 'wb') as file_handle:
     pickle.dump(all_results, file_handle)
print(f"Đã lưu thành công TOÀN BỘ kết quả vào: {save_path_pickle}")

Đã tạo Ma trận A với shape: (120, 720)
--- Solving Centralized Fair Problem (Linear) ---
★ Target Dual Optimal (V*): -79.1518
★ Optimal Allocation Shape (X*): (600, 720)
--- Start GD (10000000 iterations) | Initial Obj: 22000.17 ---
DPDS Iterations: 690
Objective value: -52.6386
DPDS Algorithm finsished at epsilon difference 0.0001 in 34.52 s

GD finished in 34.60 s
Full gradient finished in 34.60 s
Speed-up = 1.0× (wall-clock)

Đã lưu thành công TOÀN BỘ kết quả vào: /content/drive/MyDrive/EV_charging_project/experiment_result/dpds_results_600u_720i.pkl
